# Sujet 4 · Prédire la réussite à un examen : arbre de décision et régression logistique

**Cours « Intelligence artificielle »** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa

**Chapitre 3** (apprentissage supervisé, évaluer un modèle) · **Mini-projet, sujet 4** (modélisation prédictive) · **Durée :** environ 1 h 30

**Ce que vous allez faire**

- explorer les données fictives de 200 étudiants de licence de français : heures de lecture, assiduité, devoirs rendus, livres empruntés à la bibliothèque, temps de trajet, et réussite (ou non) à l'examen final ;
- entraîner deux modèles, un arbre de décision et une régression logistique, sur une partie des étudiants, puis les évaluer sur les autres (exactitude, matrice de confusion) ;
- lire l'arbre appris, puis discuter ce qu'il serait juste, ou non, de faire d'un tel modèle dans une université.

**Mode d'emploi.** Exécutez les cellules de haut en bas : cliquez sur le bouton ▶ à gauche de chaque cellule, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu « Exécution → Tout exécuter » lance tout le carnet d'un coup.

Avant chaque cellule de code, une question vous demande de **prédire** le résultat : notez votre réponse, exécutez, puis comparez avec la rubrique « Ce que vous devez voir ». Python écrit les nombres décimaux avec un point : 0.82 se lit 0,82.

> **Données fictives.** Les 200 étudiants de ce carnet sont inventés par un programme, à partir d'une graine fixe et d'une règle que nous vous montrons. Aucun étudiant réel n'est concerné : ne citez jamais ces chiffres comme un résultat.

Le programme officiel parle de « données de consommation » : ici, les emprunts de livres et le temps de lecture en tiennent lieu. La même méthode sert à prédire l'achat d'un livre ou la participation à un café littéraire, l'exemple du site du cours.

## Étape 1 · Charger les outils

Cette cellule charge NumPy, pandas, matplotlib et les outils de scikit-learn dont nous aurons besoin : séparer les données, l'arbre de décision, la régression logistique, la standardisation et les mesures d'évaluation.

**Avant d'exécuter :** combien d'outils différents viennent de scikit-learn (les lignes qui commencent par `from sklearn`) ?

In [ ]:
#@title Charger les outils (bibliothèques)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split             # séparer apprentissage et test
from sklearn.tree import DecisionTreeClassifier, plot_tree       # arbre de décision
from sklearn.linear_model import LogisticRegression              # régression logistique
from sklearn.preprocessing import StandardScaler                 # standardisation
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

COULEURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]         # couleurs des graphiques
CLASSES = ["échec", "réussite"]                                  # 0 = échec, 1 = réussite
print("Outils chargés.")

**Ce que vous devez voir :** `Outils chargés.` Cinq lignes commencent par `from sklearn`.

**Question :** la dernière de ces lignes charge trois mesures : `accuracy_score`, `precision_score` et `recall_score`. Retrouvez leur nom français dans le chapitre 3 du cours.

## Étape 2 · Fabriquer 200 étudiants fictifs

Voici le **dictionnaire des variables** de nos données :

| Variable | Signification | Valeurs |
|---|---|---|
| `heures_lecture` | heures de lecture en français par semaine, hors cours | 0 à 12 |
| `assiduite` | part des séances suivies dans le semestre | en %, de 20 à 100 |
| `devoirs` | devoirs rendus, sur les 10 demandés | 0 à 10 |
| `emprunts` | livres empruntés à la bibliothèque universitaire pendant le semestre | nombre |
| `trajet_min` | durée du trajet entre le domicile et l'université | en minutes |
| `reussite` | **la variable à prédire** : au moins 10/20 à l'examen final | 1 = oui, 0 = non |

Cette cellule tire au hasard les cinq premières variables. Le temps de trajet est tiré en premier, car un étudiant qui habite loin manque plus souvent les cours : l'assiduité baisse quand le trajet s'allonge.

Elle tire aussi une sixième variable, `travail_h`, les heures de travail salarié par semaine (petits emplois, aide à la famille). Dans notre fiction, les étudiants qui habitent loin travaillent plus souvent. Cette variable pèsera sur la réussite, mais l'université ne la connaît pas : elle **ne figurera pas** dans le tableau donné aux modèles.

**Avant d'exécuter :** d'après vous, combien d'étudiants sur 200 habitent à plus d'une heure de l'université dans ces données (le trajet va de 5 à 120 minutes) ?

In [ ]:
#@title À MODIFIER : la graine du hasard, puis exécutez (les variables explicatives)
graine = 2026 #@param {type:"integer"}
rng = np.random.default_rng(graine)      # générateur de hasard reproductible
n = 200                                  # nombre d'étudiants fictifs
trajet_min = rng.integers(5, 121, n)                                            # 5 à 120 minutes
assiduite = np.clip(rng.normal(100 - 0.3 * trajet_min, 12), 20, 100).round()    # baisse avec le trajet
heures_lecture = np.clip(rng.normal(4, 2.5, n), 0, 12).round()
devoirs = rng.binomial(10, 0.6, n)                                              # sur 10
emprunts = rng.poisson(0.5 + 0.4 * heures_lecture)                              # les lecteurs empruntent plus
travail_h = rng.poisson(trajet_min / 12)        # heures de travail salarié : INCONNUES de l'université
print("Étudiants à plus d'une heure de trajet :", (trajet_min > 60).sum())
print("Assiduité moyenne (%) :", assiduite.mean().round(1))
print("Travail salarié moyen (heures par semaine) :", travail_h.mean().round(1))

**Ce que vous devez voir :** `Étudiants à plus d'une heure de trajet : 102`, `Assiduité moyenne (%) : 80.9` et `Travail salarié moyen (heures par semaine) : 5.0`. La moitié environ des étudiants fictifs habite loin, comme dans une wilaya où beaucoup viennent de communes éloignées.

**Question :** pourquoi est-il réaliste que l'assiduité dépende du trajet ? Citez une autre variable de la vie des étudiants qui pourrait jouer le même rôle.

Il reste à fabriquer la **réussite**. La cellule suivante applique une règle cachée : un score qui augmente avec la lecture, l'assiduité, les devoirs et les emprunts, et qui baisse avec le travail salarié, puis la fonction logistique `1 / (1 + exp(−score))` qui transforme ce score en une probabilité entre 0 et 1. Enfin, le hasard décide : un étudiant qui a 70 % de chances de réussir échoue tout de même trois fois sur dix. C'est cette part de hasard qui empêchera tout modèle d'être parfait.

**Avant d'exécuter :** lisez la ligne `score = …`. Quelle variable du dictionnaire n'y figure **pas** ? Et quelle variable de la règle ne sera **pas** dans le tableau `etudiants` ? Retenez ces deux variables : nous les retrouverons dans la discussion éthique.

In [ ]:
#@title Fabriquer la réussite à l'examen (règle cachée + hasard), puis le tableau
score = -16 + 1.2 * heures_lecture + 0.13 * assiduite + 0.8 * devoirs + 0.1 * emprunts - 0.4 * travail_h
probabilite = 1 / (1 + np.exp(-score))                   # fonction logistique : entre 0 et 1
reussite = (rng.random(n) < probabilite).astype(int)     # 1 = réussite, 0 = échec
etudiants = pd.DataFrame({          # le tableau connu de l'université : sans travail_h
    "heures_lecture": heures_lecture.astype(int), "assiduite": assiduite.astype(int),
    "devoirs": devoirs, "emprunts": emprunts, "trajet_min": trajet_min, "reussite": reussite})
variables = ["heures_lecture", "assiduite", "devoirs", "emprunts", "trajet_min"]
print("Taux de réussite :", etudiants["reussite"].mean().round(2))
etudiants.head(6)

**Ce que vous devez voir :** `Taux de réussite : 0.7` (140 étudiants sur 200), puis les 6 premières lignes. Le premier étudiant lit 2 heures, suit 54 % des séances, habite à 103 minutes : il a échoué (`0`). Le quatrième (ligne 3) a réussi avec 72 % d'assiduité et 79 minutes de trajet : le hasard de la règle joue.

La variable absente de la règle est **`trajet_min`** : dans notre fiction, le trajet n'agit sur la réussite qu'**indirectement**, parce qu'il fait baisser l'assiduité et qu'il accompagne le travail salarié. Et la variable de la règle absente du tableau est **`travail_h`**.

**Question :** le quatrième étudiant a réussi alors que sa situation ressemble à celle de beaucoup d'étudiants en échec. Un modèle qui se tromperait sur lui serait-il « mauvais » ?

## Étape 3 · Explorer : ceux qui réussissent et ceux qui échouent

Avant de modéliser, on compare les deux groupes : la moyenne de chaque variable, puis une **boîte à moustaches** par variable. La boîte contient la moitié centrale des étudiants, le trait du milieu est la médiane, les « moustaches » montrent l'étendue des valeurs habituelles, les ronds isolés sont des valeurs rares.

**Avant d'exécuter :** pour quelle variable l'écart entre les deux groupes sera-t-il le plus visible ? Les étudiants en échec habitent-ils plus loin, en moyenne ?

In [ ]:
#@title Comparer les étudiants en échec (0) et en réussite (1)
print(etudiants.groupby("reussite")[variables].mean().round(1))
fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, v in zip(axes, variables):
    echecs = etudiants.loc[etudiants["reussite"] == 0, v]
    succes = etudiants.loc[etudiants["reussite"] == 1, v]
    ax.boxplot([echecs, succes])
    ax.set_xticks([1, 2], CLASSES)
    ax.set_title(v)
plt.suptitle("Chaque boîte résume un groupe d'étudiants (données fictives)")
plt.tight_layout()
plt.show()

**Ce que vous devez voir :** un tableau de moyennes et cinq paires de boîtes. En échec : 2,2 heures de lecture, 70,8 % d'assiduité, 5,3 devoirs, 1,3 emprunt, 78,5 minutes de trajet. En réussite : 4,8 heures, 85,3 %, 5,9 devoirs, 2,6 emprunts, 55,8 minutes. L'écart le plus net concerne la lecture et l'assiduité ; les devoirs diffèrent peu. Et les étudiants en échec habitent **23 minutes plus loin** en moyenne, alors que le trajet n'est pas dans la règle.

**Question :** comment expliquez-vous cet écart de trajet entre les deux groupes ? Trouvez deux raisons en relisant l'étape 2.

## Étape 4 · Séparer apprentissage et test

On n'évalue jamais un modèle sur les exemples qui ont servi à l'entraîner, pas plus qu'on ne juge un étudiant sur des questions dont il connaît les réponses par cœur. On garde donc une partie des étudiants de côté, le **jeu de test**. `stratify=y` garde le même taux de réussite dans les deux parties.

On calcule aussi l'exactitude d'un **modèle naïf**, qui prédit toujours la réponse la plus fréquente sans rien regarder : un bon modèle doit faire mieux.

**Avant d'exécuter :** avec 25 % des 200 étudiants réservés au test, combien d'étudiants serviront à l'apprentissage ? Quelle exactitude aura le modèle naïf ?

In [ ]:
#@title À MODIFIER : la part réservée au test, puis exécutez (séparation)
part_test = 0.25 #@param {type:"slider", min:0.1, max:0.5, step:0.05}
X = etudiants[variables]          # ce que le modèle regarde
y = etudiants["reussite"]         # ce qu'il doit prédire
X_app, X_test, y_app, y_test = train_test_split(X, y, test_size=part_test, random_state=0, stratify=y)
print("Étudiants pour l'apprentissage :", len(X_app))
print("Étudiants pour le test         :", len(X_test))
naif = y_test.value_counts(normalize=True).max()   # part de la réponse la plus fréquente
print("Exactitude du modèle naïf      :", round(naif, 2))

**Ce que vous devez voir :** 150 étudiants pour l'apprentissage, 50 pour le test (15 échecs et 35 réussites), et une exactitude du modèle naïf de **0.7** : en répondant toujours « réussite », on a raison pour 35 étudiants du test sur 50.

**Question :** un modèle qui obtient 72 % d'exactitude sur ces données est-il bon ? Comparez avec le modèle naïf.

## Étape 5 · L'arbre de décision

Un arbre de décision apprend une suite de questions de la forme « variable ≤ seuil ? ». C'est une base de règles SI … ALORS, comme un système expert, mais apprise à partir des exemples. La **profondeur** est le nombre maximal de questions posées à la suite.

**Avant d'exécuter :** l'arbre aura-t-il une meilleure exactitude sur les étudiants d'apprentissage (qu'il a vus) ou sur ceux du test (qu'il n'a jamais vus) ?

In [ ]:
#@title À MODIFIER : la profondeur de l'arbre, puis exécutez (arbre de décision)
profondeur = 3 #@param {type:"slider", min:1, max:8, step:1}
arbre = DecisionTreeClassifier(max_depth=profondeur, random_state=0)
arbre.fit(X_app, y_app)                         # apprentissage
pred_arbre = arbre.predict(X_test)              # prédictions pour des étudiants jamais vus
print("Exactitude sur l'apprentissage :", round(accuracy_score(y_app, arbre.predict(X_app)), 2))
print("Exactitude sur le test         :", round(accuracy_score(y_test, pred_arbre), 2))

**Ce que vous devez voir :** 0.87 sur l'apprentissage, 0.78 sur le test. L'arbre fait mieux que le modèle naïf (0.7), mais il est plus fort sur les exemples qu'il connaît que sur les nouveaux.

**Question :** que signifierait une exactitude de 1.0 sur l'apprentissage et de 0.6 sur le test ?

Cette cellule **dessine l'arbre**. Chaque case pose une question ; on descend à gauche si la réponse est « vrai », à droite si elle est « faux ». Dans chaque case : la question, `samples` (le nombre d'étudiants d'apprentissage qui arrivent là), `value` (combien d'échecs et de réussites parmi eux) et `class` (la réponse majoritaire). La couleur penche vers l'orange pour l'échec, vers le bleu pour la réussite.

**Avant d'exécuter :** quelle variable l'arbre choisira-t-il pour sa toute première question ?

In [ ]:
#@title Dessiner l'arbre de décision
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(arbre, feature_names=variables, class_names=CLASSES, filled=True, rounded=True,
          impurity=False, fontsize=9, ax=ax)
ax.set_title("L'arbre appris : à gauche si vrai, à droite si faux")
plt.show()

**Ce que vous devez voir :** la première question est `heures_lecture <= 1.5` (150 étudiants : 45 échecs, 105 réussites).

- À gauche, ceux qui lisent au plus une heure (25 étudiants). L'arbre pose alors une question sur le **trajet** : les 18 qui habitent à plus de 59,5 minutes ont tous échoué, d'où la règle SI heures_lecture ≤ 1,5 ET trajet_min > 59,5 ALORS échec. Les plus proches sont classés « réussite » si leur assiduité dépasse 75,5 %.
- À droite, ceux qui lisent au moins 2 heures (125 étudiants). SI l'assiduité est au plus de 69 % ET la lecture au plus de 5 heures, ALORS échec ; sinon, réussite.

Les deux feuilles de droite donnent la même réponse (« réussite ») : l'arbre a rendu les groupes plus homogènes sans changer sa décision.

**Question :** que pensez-vous de la règle qui utilise `trajet_min` ? Écrivez en français une autre règle SI … ALORS lue dans l'arbre : vous paraît-elle pédagogiquement plausible ?

## Étape 6 · Le surapprentissage

Plus l'arbre est profond, plus il pose de questions, et mieux il colle aux étudiants d'apprentissage, jusqu'à apprendre leurs accidents par cœur. C'est le **surapprentissage** (*overfitting*). Cette cellule entraîne un arbre pour chaque profondeur de 1 à 10 et compare les deux exactitudes.

**Avant d'exécuter :** comment évoluera la courbe « apprentissage » quand la profondeur augmente ? Et la courbe « test » ?

In [ ]:
#@title Comparer l'exactitude selon la profondeur de l'arbre (1 à 10)
lignes = []
for p in range(1, 11):
    essai = DecisionTreeClassifier(max_depth=p, random_state=0).fit(X_app, y_app)
    lignes.append({"profondeur": p, "apprentissage": accuracy_score(y_app, essai.predict(X_app)),
                   "test": accuracy_score(y_test, essai.predict(X_test))})
courbe = pd.DataFrame(lignes).round(2)
print(courbe.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(courbe["profondeur"], courbe["apprentissage"], marker="o", color=COULEURS[0], label="apprentissage")
ax.plot(courbe["profondeur"], courbe["test"], marker="s", color=COULEURS[1], label="test")
ax.axhline(naif, color="gray", linestyle="--", linewidth=1, label="modèle naïf")
ax.set(xlabel="Profondeur de l'arbre", ylabel="Exactitude")
ax.legend()
plt.show()

**Ce que vous devez voir :** l'exactitude d'apprentissage monte sans cesse, de 0.8 (profondeur 1) à 1.0 (profondeur 9 et 10) : l'arbre finit par connaître chaque étudiant par cœur. L'exactitude de test, elle, atteint au mieux 0.8 (profondeur 4), tombe à 0.66 (profondeur 5), puis stagne entre 0.74 et 0.78. L'écart grandissant entre les deux courbes est la signature du surapprentissage.

**Question :** quelle profondeur choisiriez-vous ? Pourquoi un arbre moins profond est-il aussi plus facile à expliquer à un étudiant ou à un collègue ?

## Étape 7 · La régression logistique

Malgré son nom, la régression logistique sert à **classer**. Elle cherche un poids (un **coefficient**) pour chaque variable, additionne le tout en un score, puis transforme ce score en probabilité avec la même fonction logistique qu'à l'étape 2. On standardise d'abord les variables (comme dans le carnet 02) pour que leurs coefficients soient comparables. Le standardiseur apprend la moyenne et l'écart type sur les seuls étudiants d'apprentissage.

**Avant d'exécuter :** la régression logistique fera-t-elle mieux ou moins bien que l'arbre sur le test ? Rappel : la règle cachée est elle-même une formule logistique.

In [ ]:
#@title Entraîner et évaluer une régression logistique
standardiseur = StandardScaler().fit(X_app)      # moyennes et écarts types de l'apprentissage
X_app_std = standardiseur.transform(X_app)       # variables standardisées
X_test_std = standardiseur.transform(X_test)
reglog = LogisticRegression()
reglog.fit(X_app_std, y_app)                     # apprentissage
pred_reglog = reglog.predict(X_test_std)
print("Exactitude sur l'apprentissage :", round(accuracy_score(y_app, reglog.predict(X_app_std)), 2))
print("Exactitude sur le test         :", round(accuracy_score(y_test, pred_reglog), 2))

**Ce que vous devez voir :** `Exactitude sur l'apprentissage : 0.92` et `Exactitude sur le test : 0.84`. La régression logistique fait mieux que l'arbre sur le test (0.84 contre 0.78) et bien mieux que le modèle naïf (0.7).

**Question :** avec 50 étudiants de test, un étudiant vaut 2 points d'exactitude. À combien d'étudiants correspond l'écart entre les deux modèles ? Est-ce une grande différence ?

Les **coefficients** disent dans quel sens chaque variable pousse la prédiction : positif, elle rend la réussite plus probable ; négatif, moins probable. Comme les variables sont standardisées, on peut comparer leurs tailles. Attention : un coefficient indique une **association**, pas une cause.

**Avant d'exécuter :** quel sera le plus grand coefficient ? Le coefficient de `trajet_min` sera-t-il nul, puisque le trajet n'est pas dans la règle cachée ?

In [ ]:
#@title Lire et dessiner les coefficients de la régression logistique
coefficients = pd.Series(reglog.coef_[0], index=variables).sort_values()
print(coefficients.round(2).to_string())
couleurs = [COULEURS[1] if c < 0 else COULEURS[0] for c in coefficients]   # orange si négatif
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(coefficients.index, coefficients.values, color=couleurs)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Coefficient (variables standardisées)")
ax.set_title("Sens et force de l'association avec la réussite")
plt.show()

**Ce que vous devez voir :** `trajet_min` −0.45 (seul coefficient négatif, barre orange), `emprunts` 0.33, `devoirs` 1.23, `assiduite` 1.51, `heures_lecture` 2.02 (le plus fort). Le modèle a bien retrouvé que la lecture compte le plus. Mais il donne au trajet un poids **négatif**, alors que le trajet n'est pas dans la règle cachée.

**Question :** un étudiant qui habite loin sera donc jugé moins susceptible de réussir, **à assiduité égale**. Quelle variable cachée de l'étape 2 explique ce poids ? Est-ce juste pour un étudiant qui habite loin mais ne travaille pas ?

## Étape 8 · Quelles erreurs ? Les matrices de confusion

L'exactitude ne dit pas **quelles** erreurs le modèle commet. La matrice de confusion croise la vraie réponse (lignes) et la prédiction (colonnes). Les deux cases de la diagonale sont les bonnes réponses ; les deux autres sont les erreurs : un étudiant qui a réussi mais que le modèle voyait échouer, ou un étudiant en échec que le modèle n'a pas vu venir.

**Avant d'exécuter :** les 50 étudiants du test comptent 15 échecs et 35 réussites. Quel type d'erreur sera le plus fréquent ?

In [ ]:
#@title Dessiner les matrices de confusion des deux modèles (jeu de test)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, nom, pred in zip(axes, ["Arbre de décision", "Régression logistique"], [pred_arbre, pred_reglog]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=CLASSES,
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{nom} : exactitude {accuracy_score(y_test, pred):.2f}")
    ax.set_xlabel("Prédiction du modèle")
    ax.set_ylabel("Vraie réponse")
plt.tight_layout()
plt.show()

**Ce que vous devez voir :**

- Arbre de décision : 9 échecs bien prédits, **6 échecs annoncés « réussite »**, 5 réussites annoncées « échec », 30 réussites bien prédites (9 + 30 = 39 bonnes réponses sur 50, soit 0.78).
- Régression logistique : 13 échecs bien prédits, **2 échecs annoncés « réussite »**, 6 réussites annoncées « échec », 29 réussites bien prédites (42 sur 50, soit 0.84).

Les deux modèles ne se trompent pas de la même façon : l'arbre laisse passer 6 échecs sur 15 ; la régression n'en laisse passer que 2, mais elle annonce à tort l'échec de 6 étudiants qui ont réussi.

**Question :** pour un étudiant, quelle erreur est la plus grave : être annoncé en échec alors qu'il aurait réussi, ou ne pas être repéré alors qu'il va échouer ? La réponse dépend-elle de ce que l'université fait de la prédiction ?

Si l'université veut **repérer les étudiants en difficulté**, la catégorie recherchée (la « positive », comme le spam dans le cours) est l'échec. Deux mesures du cours répondent alors à deux questions :

- le **rappel** : quelle part des étudiants réellement en échec le modèle repère-t-il ?
- la **précision** : quand le modèle annonce un échec, a-t-il raison ?

**Avant d'exécuter :** d'après les matrices, quel modèle aura le meilleur rappel pour l'échec ?

In [ ]:
#@title Rappel et précision pour la catégorie « échec » (0)
for nom, pred in [("Arbre de décision", pred_arbre), ("Régression logistique", pred_reglog)]:
    rappel = recall_score(y_test, pred, pos_label=0)
    precision = precision_score(y_test, pred, pos_label=0)
    print(nom)
    print(f"   repère {rappel:.0%} des étudiants en échec (rappel)")
    print(f"   a raison dans {precision:.0%} des cas quand il annonce un échec (précision)")

**Ce que vous devez voir :** l'arbre repère 60 % des étudiants en échec (9 sur 15) et a raison dans 64 % des cas quand il annonce un échec (9 sur 14). La régression logistique repère 87 % des échecs (13 sur 15), avec une précision de 68 % (13 sur 19) : elle trouve presque tous les étudiants en difficulté, mais près d'une fois sur trois, l'étudiant qu'elle signale aurait réussi.

**Question :** imaginez qu'on règle le modèle pour annoncer encore plus d'échecs. Le rappel monterait. Que deviendrait la précision, et qu'est-ce que cela signifierait pour les étudiants ?

## Étape 9 · Prédire pour un nouvel étudiant

Réglez les curseurs pour décrire un étudiant fictif, puis exécutez : la régression logistique donne une probabilité de réussite, l'arbre une réponse.

**Avant d'exécuter :** avec les valeurs par défaut (2 heures de lecture, 75 % d'assiduité, 6 devoirs, 1 emprunt, 60 minutes de trajet), l'étudiant a-t-il plus ou moins d'une chance sur deux de réussir selon vous ?

In [ ]:
#@title À MODIFIER : décrivez un nouvel étudiant, puis exécutez (prédiction)
heures_lecture_nouveau = 2 #@param {type:"slider", min:0, max:12, step:1}
assiduite_nouveau = 75 #@param {type:"slider", min:20, max:100, step:5}
devoirs_nouveau = 6 #@param {type:"slider", min:0, max:10, step:1}
emprunts_nouveau = 1 #@param {type:"slider", min:0, max:10, step:1}
trajet_nouveau = 60 #@param {type:"slider", min:5, max:120, step:5}
nouveau = pd.DataFrame([[heures_lecture_nouveau, assiduite_nouveau, devoirs_nouveau,
                         emprunts_nouveau, trajet_nouveau]], columns=variables)
proba = reglog.predict_proba(standardiseur.transform(nouveau))[0, 1]
print(f"Régression logistique : probabilité de réussite = {proba:.2f}")
print("Arbre de décision     :", CLASSES[arbre.predict(nouveau)[0]])

**Ce que vous devez voir :** `Régression logistique : probabilité de réussite = 0.38` et `Arbre de décision : réussite`. Les deux modèles ne sont pas d'accord ! Suivez cet étudiant dans l'arbre : il lit plus de 1,5 heure, a plus de 69 % d'assiduité et lit au plus 2,5 heures ; il arrive dans la feuille `samples = 20, value = [7, 13]`, où 13 étudiants sur 20 ont réussi.

**Question :** lequel des deux modèles croiriez-vous pour cet étudiant ? Montez seulement les heures de lecture à 4 et relancez : que deviennent la probabilité et la réponse de l'arbre ?

## Étape 10 · Discussion éthique : même étudiant, deux adresses

Reprenons l'étudiant de l'étape 9 et changeons **une seule** information : son lieu d'habitation. Le premier habite à 10 minutes de l'université, le second à 110 minutes (une commune éloignée de la wilaya). Tout le reste est identique : même lecture, **même assiduité**, mêmes devoirs.

**Avant d'exécuter :** les deux probabilités seront-elles égales ? Si elles diffèrent, de combien ?

In [ ]:
#@title Même étudiant, deux lieux d'habitation
for minutes in [10, 110]:
    etudiant = nouveau.copy()
    etudiant["trajet_min"] = minutes             # on ne change que le trajet
    p = reglog.predict_proba(standardiseur.transform(etudiant))[0, 1]
    print(f"Habite à {minutes:3d} minutes : probabilité de réussite = {p:.2f}")

**Ce que vous devez voir :** 0.55 pour l'étudiant qui habite à 10 minutes, 0.23 pour celui qui habite à 110 minutes. À comportement identique, le modèle retire 32 points de chances à l'étudiant éloigné : le premier serait prédit « réussite », le second « échec ».

Pourquoi ? Dans notre fiction, la vraie cause est le **travail salarié**, que le modèle ne voit pas. Il se rabat sur ce qui l'accompagne : le trajet. Le trajet devient une **variable « proxy »** (un indicateur de substitution). Un étudiant éloigné qui ne travaille pas est ainsi pénalisé pour une cause qui n'est pas la sienne.

**Question :** si l'université réservait le tutorat aux étudiants dont la probabilité est inférieure à 0,3, lequel des deux en bénéficierait ? Et si elle réservait une bourse d'excellence aux probabilités supérieures à 0,5 ?

### Prédire n'est pas juger : quelques repères

- **Variables « proxy ».** Le trajet n'est pas une compétence : il dit où l'on habite, donc souvent d'où l'on vient. Quand la vraie cause (ici le travail salarié) n'est pas mesurée, le modèle s'appuie sur ce qui l'accompagne, et peut ainsi pénaliser une origine géographique ou sociale sans que personne ne l'ait décidé. Retirer la colonne ne suffit pas toujours : l'assiduité, elle aussi, porte la trace du trajet. Pierre Bourdieu (*La Distinction*, 1979) a montré que les pratiques culturelles sont liées au capital scolaire et à l'origine sociale : un modèle apprend ces inégalités et risque de les reconduire.
- **Un précédent réel : l'algorithme d'Ofqual (Angleterre, 2020).** Les examens du baccalauréat anglais (*A-levels*) ayant été annulés à cause de la Covid-19, les notes ont été calculées par un algorithme qui ajustait les notes proposées par les enseignants selon les résultats passés de chaque établissement. Près de 40 % des notes ont été abaissées, surtout dans les établissements aux résultats historiquement faibles. Après des manifestations d'élèves, le gouvernement a abandonné ces notes le 17 août 2020 et est revenu aux notes des enseignants.
- **L'effet Pygmalion.** Robert Rosenthal et Lenore Jacobson (*Pygmalion in the Classroom*, 1968) ont soutenu, dans une étude célèbre et discutée, que les attentes des enseignants influencent les progrès des élèves. Une étiquette « à risque » affichée par un logiciel peut devenir une prophétie autoréalisatrice.
- **Des erreurs qui touchent des personnes.** Les matrices de confusion l'ont montré : même le meilleur modèle se trompe sur 8 étudiants du test sur 50. Derrière chaque case, il y a un étudiant mal orienté, ou oublié.
- **Le droit.** En Algérie, la loi n° 18-07 du 10 juin 2018 encadre le traitement des données à caractère personnel. Dans l'Union européenne, le règlement sur l'IA (*AI Act*, 2024) classe « à haut risque » les systèmes d'IA utilisés dans l'éducation pour décider de l'admission ou évaluer les acquis des apprenants : ils doivent être supervisés par des humains, documentés et fondés sur des données de qualité.
- **Prédire n'est pas expliquer.** Un coefficient montre une association, pas une cause : les emprunts à la bibliothèque ne font pas réussir, ils accompagnent la lecture.

## À vous

Modifiez seulement les champs, puis relancez les cellules à partir de celle que vous avez changée (dans Colab : « Exécution → Exécuter les cellules suivantes »).

1. **Étape 5 :** mettez la profondeur à 1, puis à 8. Relancez aussi le dessin de l'arbre. Lequel expliqueriez-vous à un étudiant ? Lequel est le plus exact sur le test ?
2. **Étape 4 :** passez la part du test à 0.4. Les exactitudes changent-elles beaucoup ? Qu'en concluez-vous sur la fiabilité d'un résultat obtenu avec peu d'étudiants ?
3. **Étape 9 :** trouvez le nombre d'heures de lecture à partir duquel la probabilité de réussite dépasse 0,5, puis refaites l'expérience avec un trajet de 5 minutes et de 120 minutes. Le seuil change-t-il ?

## Pour le mini-projet : vos propres données

Le sujet 4 du mini-projet propose par exemple un questionnaire qui se termine par « Avez-vous participé à un événement littéraire cette année ? (oui/non) ». Pour utiliser vos réponses :

- préparez un fichier CSV avec **une ligne par répondant**, des colonnes chiffrées (âge, livres lus par an, distance, etc.) et une colonne **codée 1 ou 0** : la variable à prédire ;
- questionnaire anonyme, rempli avec le consentement des répondants (loi n° 18-07) ; avec quelques dizaines de réponses seulement, restez très prudents dans vos conclusions ;
- écrivez le nom de la colonne à prédire dans le champ, cochez la case et exécutez : dans Colab, une fenêtre vous demande le fichier ; dans Jupyter, placez-le à côté du carnet sous le nom `mes_donnees.csv` ;
- relancez ensuite les étapes 4 à 8 (les noms « échec » et « réussite » désigneront simplement 0 et 1). Les étapes 9 et 10 ne fonctionnent qu'avec les données fictives.

**Avant d'exécuter :** si vous laissez la case décochée, que va faire la cellule ?

In [ ]:
#@title À MODIFIER : charger vos propres données (fichier CSV)
charger_mon_fichier = False #@param {type:"boolean"}
colonne_a_predire = "participe" #@param {type:"string"}
if charger_mon_fichier:
    try:
        from google.colab import files       # n'existe que dans Colab
        envoi = files.upload()               # fenêtre pour choisir le fichier
        nom_fichier = list(envoi.keys())[0]
    except ImportError:                      # Jupyter : fichier placé à côté du carnet
        nom_fichier = "mes_donnees.csv"
    etudiants = pd.read_csv(nom_fichier).dropna().rename(columns={colonne_a_predire: "reussite"})
    variables = [c for c in etudiants.select_dtypes("number").columns if c != "reussite"]
    print(len(etudiants), "réponses complètes ; variables explicatives :", variables)
else:
    print("Case non cochée : le carnet garde les données fictives.")

**Ce que vous devez voir :** avec la case décochée, `Case non cochée : le carnet garde les données fictives.` Avec votre fichier, le nombre de réponses complètes et la liste de vos variables explicatives.

**Question :** parmi les questions de votre propre questionnaire, laquelle pourrait jouer le rôle d'une variable « proxy », comme le trajet ici ?

## À rendre

Répondez par écrit, en français, en quelques lignes pour chaque question.

1. **Comparer.** Comparez l'arbre de décision, la régression logistique et le modèle naïf : exactitude, matrices de confusion, rappel et précision pour l'échec. Lequel choisiriez-vous, et pour quel usage ?
2. **Lire l'arbre.** Écrivez en phrases deux règles SI … ALORS lues dans l'arbre de profondeur 3. Sont-elles plausibles pour un enseignant de langue ? Laquelle vous surprend ?
3. **Avis éthique.** Votre université veut utiliser un tel modèle pour repérer dès novembre les étudiants « à risque ». Rédigez un avis argumenté d'une dizaine de lignes : erreurs possibles, variable du trajet, effet Pygmalion, ce qu'il faudrait exiger (supervision humaine, transparence, consentement, droit de contester).

## Ce qu'il faut retenir

- En **apprentissage supervisé**, le modèle apprend sur des exemples étiquetés (jeu d'apprentissage) et s'évalue sur des exemples jamais vus (jeu de test), en le comparant à un **modèle naïf**.
- L'**arbre de décision** apprend des règles SI … ALORS lisibles, mais **surapprend** s'il est trop profond ; la **régression logistique** donne une probabilité et des coefficients, qui indiquent une association et non une cause.
- L'exactitude ne suffit pas : la **matrice de confusion** montre quelles erreurs le modèle commet. Ces erreurs touchent des personnes : variables « proxy », étiquettes, droit des données.

**Leçons du site :** [Mini-projet, sujet 4 : modélisation prédictive](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-4) · [Chapitre 3 : évaluer un modèle, le surapprentissage](https://progremer04.github.io/canva-au-for-ang-/#c3-ml) · [Chapitre 3 : classification selon les attributs](https://progremer04.github.io/canva-au-for-ang-/#c3-algorithmes)